# Implementing Early-Exit on Resnet model

## What is Early Exit?
In recent researches, some models have been implemented that they have more than one output. Exept last one, the others are called Early-Exits. By using that, The inference time will decrease and less inference memory. In this project, I implemented EE on Resnet families. The implementation in compleatly flexible. What means that you can create EE with your costum structure and set it where do you exactly want.

### IMPORTANT
Only inference time and memory will reduce. The train time and validating time and their memory usage will defeneatly increase and the reason is related to training all neurons in the model. There for in trainging step, all data should be showen to all backbone and exits

### IMPORTANT
To exit earlier, the output should pass the treshold. Therefor, make sure the trshold has logical quantity and you have enougth exits. No more no less.

## Import libraries here
There is all libraries you need to implemet.

### NOTE
If you want to use M chipset Mack, you need to use `mps` to get access to your GPU.

In [ ]:
import numpy as np
import torch
import torch.backends
import torch.backends.mps
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets
from datasets import load_dataset
from torchvision import transforms
from torch.utils.data.sampler import SubsetRandomSampler
from tqdm import tqdm
import time

## Optional block
In some cases, during implementation of CNN models, you need to implement pooling layer right after convolutional layer. Use this block to implement this situation.

In [ ]:
class ConvPoolAc(nn.Module):
    def __init__(self, chanOut, kernel=3, stride=1, padding=1, p_ceil_mode=False, pool_pad=0):
        super(ConvPoolAc, self).__init__()

        self.layer = nn.Sequential(
            nn.LazyConv2d(chanOut, kernel_size=kernel,
                stride=stride, padding=padding, bias=False),
            nn.AvgPool2d(2, stride=2, ceil_mode=p_ceil_mode, padding=pool_pad), #ksize, stride
            nn.ReLU(True)
        )

    def forward(self, x):
        return self.layer(x)

In [ ]:
class AdvancedConvPoolAc(nn.Module):
    def __init__(self, features):
        super(AdvancedConvPoolAc, self).__init__()
        self.layer = nn.ModuleList()
        for i, item in enumerate(features):
            if item == "conv":
                conv_features = features[i + 1]
                self.layer.append(nn.LazyConv2d(conv_features["channel"],
                                                kernel_size=conv_features["kernel"],
                                                stride=conv_features["stride"],
                                                padding=conv_features["padding"], bias=False))
                self.layer.append(nn.ReLU(True))
            elif item == "pool":
                pool_features = features[i + 1]
                self.layer.append(nn.AvgPool2d(pool_features["channel"],
                                               stride=pool_features["stride"],
                                               ceil_mode=False, padding=0))

    def forward(self, x):
        for layer in self.layer:
            x = layer(x)
        return x

## Residual Blocks
Next two blocks, are residual blocks. Normal and Bottleneck form. The difference between them, are the shape of convolution layers between them. By using the bottleneck form, you can implement deeper model b ut with same number of residual blocks!!

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride = 1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size = 3, stride = stride, padding = 1)
        self.batch1 = nn.BatchNorm2d(out_channels)
        self.relu1 = nn.ReLU()
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size = 3, stride = 1, padding = 1)
        self.batch2 = nn.BatchNorm2d(out_channels)
        if in_channels != out_channels:
            self.conv3 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)
            self.batch3 = nn.BatchNorm2d(out_channels)
            shortcut = True
        else:
            shortcut = None
        self.shortcut = shortcut
        self.relu2 = nn.ReLU()
        self.out_channels = out_channels

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.batch1(out)
        out = self.relu1(out)
        out = self.conv2(out)
        out = self.batch2(out)
        if self.shortcut:
            residual = self.conv3(x)
            residual = self.batch3(residual)
        out += residual
        out = self.relu2(out)
        return out

In [ ]:
class AdvancedResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride = 1):
        super(AdvancedResidualBlock, self).__init__()
        if stride == 1.5:
            stride = 1
            f = True
        else:
            f = False
        self.conv1 = nn.Sequential(
                        nn.LazyConv2d(in_channels, kernel_size = 1, stride = stride),
                        nn.BatchNorm2d(in_channels),
                        nn.ReLU())
        self.conv2 = nn.Sequential(
                        nn.Conv2d(in_channels, in_channels, kernel_size = 3, stride = 1, padding=1),
                        nn.BatchNorm2d(in_channels),
                        nn.ReLU())
        self.conv3 = nn.Sequential(
                        nn.Conv2d(in_channels, out_channels, kernel_size = 1, stride = 1),
                        nn.BatchNorm2d(out_channels))
        self.shortcut = nn.Sequential(
                nn.LazyConv2d(out_channels, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_channels)
            )
        if stride == 1:
            self.shortcut = None
        if f:
            self.shortcut = nn.Sequential(
                nn.LazyConv2d(out_channels, kernel_size=1, stride=1, padding=0),
                nn.BatchNorm2d(out_channels)
            )
        self.relu = nn.ReLU()
        self.out_channels = out_channels

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.conv2(out)
        out = self.conv3(out)
        if self.shortcut:
            residual = self.shortcut(x)
            out += residual
        else:
            out += x
        out = self.relu(out)
        return out

## Optional but Useful
This block is related to the output layer. You sould use several outputs to implement this model. By using this layer, you can easily find the Output and implement changes more easier.

In [ ]:
class OutPutBlock(nn.Module):
    def __init__(self, in_features=720, out_features=10):
        super(OutPutBlock, self).__init__()
        self.drop = nn.Dropout(0.4)
        self.linear = nn.LazyLinear(out_features)
        self.classifier = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.drop(x)
        pred = self.classifier(self.linear(x))
        return pred

In [ ]:
class ResNet(nn.Module):
    def __init__(self):
        super(ResNet, self).__init__()
        self.backbone = nn.ModuleList()
        self.exits = nn.ModuleList()
        self.fast_inference_mode = False
        self.exit_threshold = 0.5

    def make_backbone(self, block, layers, exit_place, num_classes = 10):
        self.inplanes = 64
        back_bone_layers = []
        i = 0
        back_bone_layers.append(nn.Sequential(
                        nn.LazyConv2d(64, kernel_size = 7, stride = 2, padding = 3),
                        nn.BatchNorm2d(64),
                        nn.ReLU(),
                        nn.MaxPool2d(kernel_size = 3, stride = 2, padding = 1)))

        if block == ResidualBlock:
            for i, j in enumerate(layers):
                for k in range(j):
                    if ((i * j) + k) in exit_place:
                        self.backbone.append(nn.Sequential(*back_bone_layers))
                        back_bone_layers = []
                    if k == 0:
                        cin = pow(2, (i - 1)) * 64
                        cout = pow(2, i) * 64
                        if i == 0:
                            back_bone_layers.append(block(64, cout, stride = 1))
                        else:
                            back_bone_layers.append(block(cin, cout, stride = 2))
                    else:
                        back_bone_layers.append(block(cin, cout, stride = 1))
                    cin = cout
        elif block == AdvancedResidualBlock:
            for i, j in enumerate(layers):
                for k in range(j):
                    cin = pow(2, i) * 64
                    if ((i * j) + k) in exit_place:
                        self.backbone.append(nn.Sequential(*back_bone_layers))
                        back_bone_layers = []
                    if k == 0:
                        if i == 0:
                            back_bone_layers.append(block(cin, 4 * cin, stride = 1.5))
                        else:
                            back_bone_layers.append(block(cin, 4 * cin, stride = 2))
                    else:
                        back_bone_layers.append(block(cin, 4 * cin, stride = 1))

        self.backbone.append(nn.Sequential(*back_bone_layers))

    def make_exits(self, exit_place, num_classes, exit_mods=None):
        for i in exit_place:

            if i == 0:
                self.exits.append(nn.Sequential(
                    nn.Sequential(
                    ConvPoolAc(chanOut=32, kernel=3, stride=1, padding=1),
                    ConvPoolAc(chanOut=16, kernel=3, stride=1, padding=1)),
                    nn.Sequential(nn.LazyLinear(512),
                                  OutPutBlock(512, num_classes))
                    ))
            else:
                if i in exit_mods.keys() and exit_mods[i] != []:
                    self.exits.append(nn.Sequential(
                        AdvancedConvPoolAc(exit_mods[i]),
                        OutPutBlock(512, num_classes)
                        ))
                else:
                    self.exits.append(nn.Sequential(
                        nn.AvgPool2d(kernel_size=2, stride=2),
                        OutPutBlock(512, num_classes)
                        ))
        self.exits.append(nn.Sequential(
            nn.AvgPool2d(kernel_size=2, stride=2),
            OutPutBlock(512, num_classes)
            ))

    def exit_criterion_top1(self, x):
        with torch.no_grad():
            # pk = nn.functional.softmax(x, dim=-1)
            top1 = torch.max(x) #x)
            return top1 > self.exit_threshold

    def _forward_training(self, x):
        res = []
        for backbone, current_early_exit in zip(self.backbone, self.exits):
            x = backbone(x)
            y = current_early_exit[0](x)
            y = y.view(y.size(0), -1)
            res.append(current_early_exit[1](y))
        return res

    def forward(self, x):

        if self.fast_inference_mode:
            i = 1
            for backbone, current_early_exit in zip(self.backbone, self.exits):
                x = backbone(x)
                res = current_early_exit[0](x)
                res = res.view(res.size(0), -1)
                res = current_early_exit[1](res)
                if self.exit_criterion_top1(res):
                    return res, i
                i += 1
            return res, 0
        else:
            return self._forward_training(x)

    def set_inference_parameters(self, mode=True, thresh=0.5):
        self.fast_inference_mode = mode
        self.exit_threshold = thresh